# AR-only ablation training run

Standalone companion to `train_recognizer_v2_scratch.ipynb` (the full two-head
production run) and to `train_recognizer_ctc_only.ipynb` (its CTC-only twin).
Works unmodified on **Colab**, **Kaggle**, and a **local machine** -- on **CPU,
GPU, or TPU** (auto-detected):
- Colab: mounts Google Drive at `/content/drive/My Drive/tuna-ocr` and checkpoints there.
- Kaggle: checkpoints to `/kaggle/working/tuna-ocr` (persisted as notebook output).
- Local: checkpoints to `recognizer/checkpoints/` in the repo.

Reuses the exact same data pipeline (pull/pack/dedup, tokenizer-decode patch,
low-memory dataset-load patch, newline data-quality filter) as
`train_recognizer_v2_scratch.ipynb` -- sections 1-2 below are copied verbatim
from it, so see that notebook for the reasoning behind each. Section 3 is
where this notebook diverges.

Training logs every 100 steps and pushes a checkpoint every 2,000 steps to
this run's own private Hugging Face repo (see section 3).

**Before running:** add an `HF_TOKEN` secret (Colab: key icon in the left
sidebar; Kaggle: Add-ons > Secrets; local: `export HF_TOKEN=hf_...`).

**Platform settings to check first:**
- **Kaggle**: internet access is off by default -- Settings (right sidebar) >
  Internet > On. Also set Settings > Accelerator to GPU/TPU if you want one.
- **Colab**: Runtime > Change runtime type > pick GPU or TPU if you want one
  (default is CPU-only).


## This notebook: AR-only ablation (80,000 steps, sequential mode)

Trains **only the AR decoder** -- `run_training(..., train_ctc=False,
train_ar=True)`. The CTC head is excluded from every step ENTIRELY, not
just zero-weighted: `model.forward` skips its linear layer, `compute_loss`
never computes a CTC loss term, and both the periodic eval and the sample
log skip its CER/preview text -- see `recognizer/train.py`'s `train_ctc`
docstring. `TrainConfig.filter_unlearnable` is also off here (it drops
samples whose CTC target is longer than the encoder can frame, which is
meaningless when CTC never trains, and would otherwise cost this run real
training samples for no reason).

**Mode**: trains in **sequential** AR mode for its entire budget
(`sequential_ar_steps=80_000`, i.e. >= `max_steps` -- see `modules/decoder.py`'s
"Two-stage training" note), not blockwise. Sequential mode is plain
one-token-at-a-time teacher forcing with unrestricted cross-attention -- the
mode that reached 0.09 `val_ar_cer` on the `v2_scratch_k16` run before it
switched to blockwise at step 60k. This isolates the decoder from BOTH CTC's
competing gradient AND the still-unfixed blockwise `split_into_blocks`
target-mismatch issue (see that run's "Fix 3" markdown cell), for the
cleanest possible read on how good this decoder architecture can get.

**Tokenizer**: character-level (`recognizer/tokenizer/char_tokenizer.py`'s
`CharTokenizer`), shared with the companion `train_recognizer_ctc_only.ipynb`
via `run_training(..., unify_ctc_tokenizer=True)`, instead of the production
run's SentencePiece subword tokenizer -- so both ablation notebooks' target
label space is directly comparable. Character-level AR targets are longer per
line than subword targets, so `val_ar_cer` here is NOT directly comparable to
`v2_scratch_k16`'s subword-based `val_ar_cer` in absolute terms -- compare
against the CTC-only notebook's numbers, and against the production run's
*shape* (does it plateau the same way, or keep improving?), not its exact
value.

**Budget**: 80,000 steps. **Identity**: fresh weights,
`RUN_NAME = "v2_ar_only"`. Checkpoints push to `Panhapich/tuna-ocr` -- the
SAME repo `train_recognizer_v2.ipynb` ("v1") pushes to at the repo ROOT, and
shared with the companion `train_recognizer_ctc_only.ipynb` too -- but this
run pushes under its own folder (`ar_only/step_XXXXXXX.pt`) via
`run_training(..., hub_path_prefix="ar_only")`, and `pull_latest_checkpoint(
..., path_prefix="ar_only")` only ever looks inside that folder when
resuming -- see `recognizer/hf_push.py`'s `path_prefix` docstring. This
keeps this run's checkpoints from ever colliding with "v1"'s root-level
ones or with the CTC-only notebook's `ctc_only/` folder.


In [1]:
import os, subprocess, sys

def detect_environment():
    # Kaggle is checked first: some Kaggle kernels leak a stray COLAB_GPU/
    # COLAB_RELEASE_TAG env var, which would otherwise misdetect as Colab and
    # crash trying to mount Google Drive. "/kaggle/working" existing is a much
    # harder signal to spoof than an env var, so it takes priority.
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"
REPO_DIR = "tuna-ocr"

def run_git(args):
    """Runs git and raises with git's ACTUAL stderr on failure. A bare
    CalledProcessError only reports "exit status 128", which is git's catch-all
    and says nothing about which of the many possible causes (existing
    directory, auth, network) actually happened."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed ({r.returncode}):\n{r.stderr.strip()}")
    return r

# Three cases, in order. The middle one is the important fix: after a kernel
# restart the cwd resets to /content (or /kaggle/working), so "recognizer" is no
# longer visible even though a previous run already cloned the repo -- the old
# code then tried to clone again and git aborted with "destination path already
# exists and is not an empty directory" (exit 128).
def pull_latest():
    """Fast-forward the clone we're standing in, LOUDLY. A stale clone is the
    single most confusing failure mode of this notebook: the library code is
    older than the notebook cell driving it, so you get a TypeError about an
    unexpected keyword argument for a config field that plainly exists on
    GitHub. Failing to pull is survivable (offline runtime, dirty tree), so
    this doesn't raise -- but it must never be a quiet one-line note."""
    try:
        run_git(["pull", "--ff-only"])
        print("pulled latest changes")
    except RuntimeError as e:
        print("!" * 78)
        print("WARNING: could not update the clone -- running POSSIBLY STALE code.")
        print(f"  {e}")
        print("  If a later cell fails with 'unexpected keyword argument', this is why.")
        print("  Fix: !git -C . fetch origin && git -C . reset --hard origin/main")
        print("       then restart the runtime (stale modules stay imported).")
        print("!" * 78)

if os.path.isdir("recognizer"):
    # Already inside the repo -- which is what re-running this cell in the same
    # session always looks like, since the first run chdir'd here. This branch
    # used to just print and return, so a second run silently kept whatever code
    # the session started with and never saw upstream commits again.
    print(f"already inside the repo working dir: {os.getcwd()}")
    pull_latest()
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"found an existing clone, reusing it: {os.getcwd()}")
    pull_latest()
else:
    run_git(["clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    print(f"cloned into {os.getcwd()}")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())
# Print the resolved commit: the one unambiguous answer to "is my library code
# actually the version I think it is?", checkable against the GitHub history.
print("repo commit:  " + run_git(["log", "-1", "--pretty=%h %s"]).stdout.strip())


environment: colab
cloned into /content/tuna-ocr
working dir: /content/tuna-ocr
repo commit:  793afa0 Add CTC-only/AR-only ablation notebooks and Hub checkpoint cleanup script


In [2]:
# Colab and Kaggle both ship torch preinstalled and matched to their runtime (the CUDA
# driver on a GPU runtime, or the torch_xla/libtpu build on a TPU runtime) -- blindly
# `pip install torch` on top of that (e.g. via a plain `-r recognizer/requirements.txt`)
# can silently replace it with a build that doesn't match, which breaks GPU support and
# breaks TPU support even harder (torch_xla is pinned to one exact torch version).
# Install everything else normally, and only pip-install torch if it isn't importable
# at all (a bare local venv).
import importlib.util
from pathlib import Path

def strip_torch(req_path):
    lines = Path(req_path).read_text().splitlines()
    return [l for l in lines if not l.strip().lower().startswith("torch")]

torch_before = None
if importlib.util.find_spec("torch") is not None:
    import torch
    torch_before = torch.__version__

reqs = strip_torch("recognizer/requirements.txt") + strip_torch("real_data/requirements.txt")
Path("/tmp/_notebook_requirements.txt").write_text("\n".join(reqs) + "\n")
!pip install -q -r /tmp/_notebook_requirements.txt

if torch_before is None:
    print("torch not found -- installing (no preinstalled build to preserve here)")
    !pip install -q torch
else:
    # Excluding torch from the requirements file isn't a complete guarantee: any
    # dependency in it is free to pull a *different* torch in as its own dependency.
    # On a TPU runtime that's silently fatal -- torch_xla only loads against the exact
    # torch build it was compiled for, and the failure surfaces much later as an opaque
    # import/libtpu error, so check explicitly here rather than discovering it then.
    # importlib.metadata, not `torch.__version__`: torch is already imported in this
    # kernel, so its module object still reports the OLD version no matter what pip
    # just wrote to disk (and importlib.reload(torch) is not a safe way to find out).
    # The distribution metadata reflects what's actually installed now.
    from importlib.metadata import version as _pkg_version
    torch_after = _pkg_version("torch")
    if torch_after != torch_before:
        print(f"WARNING: pip changed torch {torch_before} -> {torch_after} as a "
              f"transitive dependency. On a TPU runtime, restart the runtime and "
              f"`pip install torch=={torch_before}` before continuing, or torch_xla "
              f"will fail to load.")
    else:
        print(f"using preinstalled torch {torch_after} "
              f"(cuda available: {torch.cuda.is_available()}) -- not reinstalled")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.6/541.6 kB 11.9 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 99.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 86.4 MB/s eta 0:00:00
using preinstalled torch 2.11.0+cu128 (cuda available: True) -- not reinstalled


In [3]:
import os

from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)

# Token resolution, in order. Colab's own secret store (env_utils.get_hf_token) is
# tried first but is NOT reliable: it raises "Secrets can only be fetched when running
# from the Colab UI" whenever the notebook runs detached from the UI tab, which is
# exactly what happened on a long training run here. So fall back to an HF_TOKEN
# environment variable, then to a plain file, then to an interactive prompt --
# deliberately never hardcoded in this notebook, which is committed to a public git
# repo (GitHub's push protection rejects the commit outright, and HF's secret scanner
# auto-revokes any write-scoped token that lands in one).
#
# Easiest on Colab: run this in a scratch cell once per session, paste when prompted:
#     import os, getpass; os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")
hf_token = None
try:
    hf_token = env_utils.get_hf_token(ENV)
except Exception as e:
    print(f"platform secret store unavailable ({type(e).__name__}), trying fallbacks...")
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    for candidate in ("/content/hf_token.txt", "/kaggle/working/hf_token.txt", "hf_token.txt"):
        if os.path.exists(candidate):
            hf_token = open(candidate).read().strip()
            print(f"read HF token from {candidate}")
            break
if not hf_token:
    import getpass
    hf_token = getpass.getpass("HF token (input hidden): ").strip()

# Resolve the accelerator NOW, before the multi-hour cells below, and print the exact
# torch device that training will use. detect_accelerator() covers both TPU
# generations (legacy XRT env vars and current PJRT ones) plus the /dev/accel* device
# nodes, so a modern Colab/Kaggle TPU runtime is recognised rather than falling through
# to CPU -- a fallback that is otherwise invisible until you notice steps taking 100x
# too long, hours in.
accelerator = env_utils.detect_accelerator()
device = env_utils.get_torch_device()

print("environment:      ", ENV)
print("checkpoint root:  ", checkpoint_root)
print("HF token loaded:  ", bool(hf_token))
print("accelerator:      ", env_utils.describe_accelerator())
print("torch device:     ", device)
if accelerator == "cpu":
    print("\n>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. "
          "Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- "
          "at this dataset's scale it will not finish.")


Mounted at /content/drive
platform secret store unavailable (RuntimeError), trying fallbacks...
environment:       colab
checkpoint root:   /content/drive/My Drive/tuna-ocr/checkpoints
HF token loaded:   True
accelerator:       cuda: Tesla T4 (15.6 GB)
torch device:      cuda


In [4]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetched Panhapich/khmer-sp-8k -> /content/tuna-ocr/recognizer/tokenizer/assets: ['gazetteer.json', 'khmer_segmentation.py', 'khmer_sp.model', 'latin_exceptions.json', 'tokenizer_info.json']


PosixPath('/content/tuna-ocr/recognizer/tokenizer/assets')

## 1a. Patch a tokenizer decode bug (inflates val_ar_cer)

The vendored `khmer_segmentation.py` (downloaded above from `Panhapich/khmer-sp-8k`)
has a `KhmerTokenizer.decode()` that does `self.sp.decode(ids).replace(" ", "")` --
this strips **every** space unconditionally, not just the artificial Khmer
word-boundary spaces `segment_line()` introduces for SentencePiece training. So any
real space in the reference text (English words, mixed Khmer/English text,
punctuation spacing) is deleted from the AR decoder's decoded output regardless of
whether the model predicted it correctly -- inflating `val_ar_cer` for any sample
containing genuine whitespace. `val_ctc_cer` is unaffected (`CharVocab.decode` in
`recognizer/data/char_vocab.py` does no space stripping), so it remains a trustworthy
read on encoder accuracy even without this patch.

This cell rewrites the downloaded `khmer_segmentation.py` on disk (the same file
every `KhmerOcrTokenizer()` construction loads from -- see
`khmer_ocr_tokenizer.py`'s `_load_upstream_khmer_tokenizer`) so `decode()` only
strips a space when it falls strictly between two Khmer characters -- the actual
artificial-boundary case -- and leaves every other space alone. Idempotent: skips if
already patched, so re-running this cell (or a fresh fetch that re-downloads the
original file) is safe.

In [5]:
from recognizer.config import TOKENIZER_ASSETS_DIR

_seg_path = TOKENIZER_ASSETS_DIR / "khmer_segmentation.py"
_seg_src = _seg_path.read_text(encoding="utf-8")

_BUGGY_DECODE = '''    def decode(self, ids) -> str:
        # Strip the artificial word-boundary spaces introduced for training;
        # natural Khmer orthography does not space every word.
        return self.sp.decode(ids).replace(" ", "")'''

_PATCHED_DECODE = '''    def decode(self, ids) -> str:
        # PATCHED (notebook cell 1a): the original body here did
        # `self.sp.decode(ids).replace(" ", "")`, which strips EVERY space --
        # including genuine ones in English words, mixed Khmer/English text, and
        # punctuation spacing -- not just the artificial Khmer word-boundary
        # spaces segment_line() introduces for SentencePiece training. That
        # silently deletes correctly-predicted spaces before CER ever sees them.
        # Only strip a space strictly between two Khmer characters -- the real
        # artificial-boundary case -- and leave every other space alone.
        import re as _re
        decoded = self.sp.decode(ids)
        return _re.sub(r"(?<=[\\u1780-\\u17ff])\\s(?=[\\u1780-\\u17ff])", "", decoded)'''

if _PATCHED_DECODE in _seg_src:
    print(f"{_seg_path} already patched -- nothing to do")
elif _BUGGY_DECODE in _seg_src:
    _seg_path.write_text(_seg_src.replace(_BUGGY_DECODE, _PATCHED_DECODE), encoding="utf-8")
    print(f"patched {_seg_path}: decode() now only strips Khmer-Khmer boundary spaces")
else:
    raise RuntimeError(
        f"{_seg_path} doesn't match the expected buggy decode() body -- the upstream "
        f"file may have changed. Inspect it manually before training: the goal is a "
        f"decode() that doesn't strip every space unconditionally."
    )


patched /content/tuna-ocr/recognizer/tokenizer/assets/khmer_segmentation.py: decode() now only strips Khmer-Khmer boundary spaces


## 1b. Low-memory dataset loading patch

`recognizer/data/manifest.py`'s `load_dedup_arrow` (unmodified library code)
materializes the ENTIRE image-bytes column into a Python list
(`table.column("image").to_pylist()`), then builds a second full list of
`Sample` objects from it -- both lists stay alive simultaneously until the
function returns, so peak memory during dataset loading is roughly **2x**
the dataset's actual image-bytes size. On a Colab session this is enough to
get the kernel OOM-killed mid-load, which surfaces as no Python traceback at
all -- just `"Canceled future for execute_request message before replies
were done"` -- because the process itself dies, not one call inside it.

This cell monkeypatches `load_dedup_arrow` to iterate the Arrow columns
directly instead of pre-snapshotting them, so no intermediate full-column
Python list is ever alive alongside the final result -- same output, roughly
half the peak memory. This is a real fix to shared library code, not a
notebook-only workaround -- worth upstreaming into
`recognizer/data/manifest.py` directly once confirmed, so every consumer of
`run_training` benefits, not just this notebook.

In [6]:
# Monkeypatches recognizer.data.manifest.load_dedup_arrow: safe because
# load_dedup_manifest (which run_training actually calls) looks up
# load_dedup_arrow by name in the module's own namespace at CALL time, not at
# import time -- so reassigning the module attribute here takes effect for
# every call made after this cell runs, without needing to touch train.py or
# re-import anything downstream.
import recognizer.data.manifest as _manifest

def _load_dedup_arrow_low_memory(path):
    import pyarrow as pa

    with pa.memory_map(str(path), "rb") as source:
        table = pa.ipc.open_file(source).read_all()
    text_col = table.column("text")
    source_col = table.column("source")
    image_col = table.column("image")
    # zip() over ChunkedArrays iterates chunk-by-chunk, yielding pa.Scalar
    # objects one at a time -- .as_py() converts just that one value, so at
    # most one row's worth of extra Python objects exists beyond the `samples`
    # list actually being built, vs. the original's three full-column lists
    # PLUS the final list all alive at once.
    samples = []
    for t, s, img in zip(text_col, source_col, image_col):
        samples.append(_manifest.Sample(image_bytes=img.as_py(), text=t.as_py(), source=s.as_py()))
    return samples

_manifest.load_dedup_arrow = _load_dedup_arrow_low_memory
print("patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading")

patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading


## 2. Data

The full pull -> pack -> dedup pipeline below is expensive (real network transfer +
CPU-bound hashing, potentially a long time at this scale) -- it only needs to run
**once**. The first successful run pushes its result to a private Hugging Face
dataset repo (`real_data.config.HF_DATA_REPO_ID`); every later run (new session, new
notebook, different machine) checks that repo first and just downloads the prebuilt
`dedup.arrow` instead of repeating the pull/pack/dedup work from scratch.

`SAMPLES_PER_SOURCE` only matters the first time (before anything's been pushed to the
Hub). The defaults pull everything available from the three smaller sources, but cap
`chanrith_ocr_image_line` (12M+ rows, ~40GB in full) at 100k rows. Each source is
pulled, packed into a single Arrow file (`<source>.arrow`, image bytes stored exactly
as pulled -- no re-encoding/resizing), and its raw per-image files are deleted before
the next source starts, bounding peak disk usage to "one source's raw files + all
Arrow files packed so far."


In [7]:
import os, shutil, subprocess, sys
from pathlib import Path
from real_data.config import EXTERNAL_DATASETS, HF_DATA_REPO_ID, REAL_DATA_ROOT
from real_data import hf_push

# Per-source sample counts for the (one-time) pull from source. Pulls everything
# available from the smaller sources, but caps chanrith_ocr_image_line (12M+ rows) at
# 100k -- pulling it in full would be ~40GB, far more than a Kaggle/Colab session's
# disk budget can hold.
SAMPLES_PER_SOURCE = {
    "deepcopy_khmer_text_recognition": 136_117,
    "chanrith_ocr_image_line": 100_000,
    "darayut_scene_text": 102_500,
    "sokheng_synthetic_v1": 100_000,
}

# KMP_DUPLICATE_LIB_OK/OMP_NUM_THREADS: Colab/Kaggle commonly have more than one
# OpenMP runtime on the import path (numpy, PIL/imagehash, datasets' native deps each
# bundle their own libomp/libiomp5) -- loading two in one process is a well-known cause
# of an immediate SIGABRT with zero output, right at import time, before any of this
# script's own code runs. Setting these before spawning avoids that class of crash;
# harmless if it wasn't actually the cause.
SUBPROCESS_ENV = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE", "OMP_NUM_THREADS": "1"}

def run_checked(cmd):
    """Runs `cmd`, always printing its output, and raises with the actual captured
    stderr on failure -- a bare `subprocess.CalledProcessError` (or, worse, a `!shell`
    cell whose exit code isn't checked at all) hides exactly the text that explains
    *why* it died, which is the difference between a one-line fix and a guessing game."""
    result = subprocess.run(cmd, env=SUBPROCESS_ENV, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        sys.stderr.write(result.stderr)
        raise RuntimeError(
            f"command failed (exit code {result.returncode}"
            f"{', likely killed by a signal -- see stderr above for the real cause' if result.returncode < 0 else ''}"
            f"): {' '.join(cmd)}"
        )
    return result

dedup_manifest = REAL_DATA_ROOT / "samples" / "dedup.arrow"
built_locally = False  # tracks whether THIS run built dedup_manifest from source
                        # (vs. it already being local, or downloaded from the Hub) --
                        # only push to the Hub in the first case (section 2b below).

if dedup_manifest.exists():
    print(f"{dedup_manifest} already present locally, skipping pull/download")
elif hf_push.dataset_exists_on_hub(HF_DATA_REPO_ID, token=hf_token):
    print(f"found a prebuilt dataset on the Hub ({HF_DATA_REPO_ID}) -- downloading "
          f"instead of re-pulling/re-deduplicating from source")
    hf_push.pull_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID)
else:
    print(f"no prebuilt dataset found on {HF_DATA_REPO_ID} -- pulling + packing from "
          f"source (one-time cost; result gets pushed to the Hub in the next cell)")
    built_locally = True

    # Pull -> pack to Arrow -> delete raw, one source at a time (not all sources
    # pulled first, then packed): this bounds peak disk usage to "current source's
    # raw files + every Arrow file packed so far", instead of needing all 4 sources'
    # raw files on disk simultaneously.
    arrow_files = []
    for source in EXTERNAL_DATASETS:
        arrow_path = REAL_DATA_ROOT / "samples" / f"{source}.arrow"
        arrow_files.append(arrow_path)
        if arrow_path.exists():
            print(f"{source}: already packed, skipping")
            continue

        source_dir = REAL_DATA_ROOT / "samples" / source
        num_samples = SAMPLES_PER_SOURCE[source]
        if not (source_dir / "manifest.tsv").exists():
            print(f"{source}: pulling {num_samples} samples...")
            run_checked([sys.executable, "-m", "real_data.generate_external_chunks",
                         "--source", source, "--num-samples", str(num_samples)])

        print(f"{source}: packing to {arrow_path}...")
        run_checked([sys.executable, "-m", "real_data.pack_arrow",
                     "--source", source, "--delete-raw"])

    print(arrow_files)


found a prebuilt dataset on the Hub (Panhapich/tuna-ocr-data) -- downloading instead of re-pulling/re-deduplicating from source


dedup.arrow: reconstructing file:   0%|          |  0.00B / 3.95GB            

dedup.arrow: downloading bytes:           |  0.00B            

In [8]:
# 2b. Deduplicate + push to the Hub -- only runs if this session actually built the
# dataset from source above (built_locally == True); a no-op if dedup_manifest was
# already local or was just downloaded from the Hub.
if built_locally:
    # --near-dup-threshold 0 disables the O(n^2) near-dup pass -- REQUIRED at this
    # scale (hundreds of thousands of rows): the default pairwise comparison is
    # O(n^2) and would take an impractically long time (the nonzero default is only
    # tuned/safe for the notebook-scale hundreds-to-thousands range, e.g.
    # notebooks/train_diagnose_eval.ipynb).
    missing = [str(p) for p in arrow_files if not p.exists()]
    if missing:
        raise RuntimeError(
            "The following sources are missing their packed .arrow file -- re-run "
            "the pull cell above (in full, for all 4 sources) before deduplicating. "
            "This usually means the runtime restarted/reset between the pull and "
            "dedup cells (e.g. after a crash) and the previously-pulled data under "
            f"{REAL_DATA_ROOT} was lost:\n  " + "\n  ".join(missing)
        )

    run_checked([sys.executable, "-m", "real_data.deduplicate",
                 "--arrow-files", *[str(p) for p in arrow_files],
                 "--out", str(dedup_manifest),
                 "--near-dup-threshold", "0"])
    assert dedup_manifest.exists(), (
        f"{dedup_manifest} was not created -- check the pull cell above actually "
        f"populated {[str(p) for p in arrow_files]} before dedup ran."
    )
    print("dedup arrow file ready:", dedup_manifest)

    print(f"pushing prebuilt dataset to the Hub ({HF_DATA_REPO_ID}) so future runs "
          f"can skip straight to downloading it...")
    url = hf_push.push_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID, private=True)
    print("pushed:", url)
else:
    print(f"{dedup_manifest} already ready (local or from the Hub) -- nothing to dedup/push")


/content/tuna-ocr/real_data/samples/dedup.arrow already ready (local or from the Hub) -- nothing to dedup/push


## 2c. Data quality filter

One of the three fixed diagnostic samples tracked during a prior training run
(ground truth `'វិរាគចិត្ត ២០០២៛\nជោះ 8x10,000'`, with a literal newline in the
transcript) produced garbage predictions across 8,000+ straight logged steps --
a strong sign the image is a single cropped line but the transcript spans two,
not something any amount of training fixes. `find_unlearnable` (run inside
`run_training`, next section) only drops samples whose CTC target is longer
than the encoder frames the image can produce -- it does not catch this
different failure mode, where the transcript simply does not correspond to
what's pictured.

This cell scans `dedup.arrow` for embedded newlines (the cheapest,
highest-confidence signal available without per-sample manual review) and
writes a filtered copy, `dedup_filtered.arrow`, that the training cell uses
instead. Runs once and is cached like every other data-prep step in this
notebook -- if you add other mismatch heuristics later, delete
`dedup_filtered.arrow` to force a rebuild.

In [9]:
import pyarrow as pa

dedup_filtered = dedup_manifest.parent / "dedup_filtered.arrow"

# Validity, not just existence: a prior interrupted write (Colab disconnect,
# kernel restart, out-of-memory mid-write) can leave a truncated/corrupt file
# behind, and pa.ipc.open_file on a corrupt file raises "ArrowInvalid: Not an
# Arrow file" much later, inside run_training -- confusing, since by then it
# looks like a training-cell bug rather than a leftover bad file from this
# cell. Checking exists() alone (as an earlier version of this cell did) trusts
# that leftover file forever, since it never gets rewritten once present.
def _is_valid_arrow_file(path):
    if not path.exists():
        return False
    try:
        with pa.memory_map(str(path), "rb") as f:
            pa.ipc.open_file(f).schema
        return True
    except pa.ArrowInvalid:
        return False

if _is_valid_arrow_file(dedup_filtered):
    print(f"{dedup_filtered} already present and valid, skipping filter pass")
else:
    if dedup_filtered.exists():
        print(f"{dedup_filtered} exists but is not a valid Arrow file "
              f"(likely an interrupted write from a previous session) -- rebuilding")
    print(f"scanning {dedup_manifest} for transcript/image line-count mismatches...")
    with pa.memory_map(str(dedup_manifest), "rb") as source:
        table = pa.ipc.open_file(source).read_all()

    texts = table.column("text").to_pylist()
    sources = table.column("source").to_pylist()

    # A literal "\n" in a transcript is the cheapest, highest-confidence signal
    # that the label spans more lines than the (single-line-cropped) image
    # actually shows -- no gradient update can fix a target that doesn't match
    # its image. Extend this predicate if other mismatch patterns turn up.
    keep_mask = [("\n" not in t) for t in texts]
    dropped_by_source = {}
    for t, s, keep in zip(texts, sources, keep_mask):
        if not keep:
            dropped_by_source[s] = dropped_by_source.get(s, 0) + 1

    n_total = len(texts)
    n_dropped = n_total - sum(keep_mask)
    print(f"dropping {n_dropped}/{n_total} samples with embedded newlines "
          f"(likely multi-line transcript vs single-line image): {dropped_by_source or 'none'}")

    filtered_table = table.filter(pa.array(keep_mask))
    # Write to a .tmp path and atomically rename into place on success -- same
    # pattern dataset.py's compute_widths uses for its widths_cache.json, so an
    # interrupted write (Colab disconnect, OOM, kernel restart) never leaves a
    # half-written dedup_filtered.arrow sitting at the real filename for a later
    # run to mistake for a finished, valid file.
    tmp_path = dedup_filtered.with_name(dedup_filtered.name + ".tmp")
    with pa.OSFile(str(tmp_path), "wb") as sink:
        with pa.ipc.new_file(sink, filtered_table.schema) as writer:
            writer.write_table(filtered_table)
    tmp_path.replace(dedup_filtered)
    print(f"wrote filtered dataset -> {dedup_filtered}")

dedup_manifest = dedup_filtered
print("training will use:", dedup_manifest)

scanning /content/tuna-ocr/real_data/samples/dedup.arrow for transcript/image line-count mismatches...
dropping 23999/428911 samples with embedded newlines (likely multi-line transcript vs single-line image): {'sokheng_synthetic_v1': 23999}
wrote filtered dataset -> /content/tuna-ocr/real_data/samples/dedup_filtered.arrow
training will use: /content/tuna-ocr/real_data/samples/dedup_filtered.arrow


## 3. Train (AR-only, sequential mode, 80,000 steps)


In [10]:
from pathlib import Path

from recognizer.config import ModelConfig, TrainConfig
from recognizer.train import run_training
from recognizer.hf_push import pull_latest_checkpoint

# max_tokens_per_block is irrelevant to how this run actually trains
# (sequential_ar_steps below keeps it in sequential mode -- plain flat-sequence
# teacher forcing -- for the entire run, never touching the blockwise
# ar_target/block_head at all). Left generous purely so data/dataset.py's
# split_into_blocks (which still runs on every sample regardless of mode)
# doesn't spam truncation warnings for the character-level target sequences --
# pure log-noise suppression, not a training-quality knob here.
model_cfg = ModelConfig(max_tokens_per_block=64)

LOG_EVERY = 100
NUM_WORKERS = 0
CKPT_EVERY = 2_000

train_cfg = TrainConfig(
    log_every=LOG_EVERY,
    num_workers=NUM_WORKERS,
    ckpt_every=CKPT_EVERY,
    max_eval_samples=512,   # unused when train_ctc=False below (evaluate_val_cer
                             # skips the whole CTC CER pass) -- harmless to leave set.
    max_ar_eval_samples=64,
    max_steps=80_000,
    # CTC is excluded from every training/eval samples whose CTC target would be
    # too long for the encoder to frame -- irrelevant when CTC never trains, and
    # would otherwise cost this run real samples for no reason (see the
    # markdown above).
    filter_unlearnable=False,
    sequential_ar_steps=80_000,   # >= max_steps above, so ar_mode never switches to
                                   # blockwise -- sequential-AR for the entire budget.
)

RUN_NAME = "v2_ar_only"
CHECKPOINT_REPO_ID = "Panhapich/tuna-ocr"  # SAME repo as train_recognizer_v2.ipynb ("v1") and both ablation notebooks
HUB_PATH_PREFIX = "ar_only"  # this run's own folder within that shared repo

# RESUME: always from the Hub, never from a local last.pt -- same reasoning as
# train_recognizer_v2_scratch.ipynb's training cell (see its comment): the Hub
# is the one copy every session, past and present, agrees on.
resume_path = pull_latest_checkpoint(Path(checkpoint_root) / RUN_NAME, token=hf_token,
                                     repo_id=CHECKPOINT_REPO_ID, path_prefix=HUB_PATH_PREFIX)
if resume_path:
    print(f"resuming from the latest checkpoint on the Hub: {resume_path}")
else:
    print(f"no Hub checkpoint found for {CHECKPOINT_REPO_ID} -- starting {RUN_NAME} from scratch")

model = run_training(
    model_cfg, train_cfg,
    dedup_manifest_path=dedup_manifest,  # points at dedup_filtered.arrow -- see section 2c.
    checkpoint_root=checkpoint_root,
    run_name=RUN_NAME,
    push_to_hub=True,
    repo_id=CHECKPOINT_REPO_ID,
    hf_token=hf_token,
    hub_private=True,
    auto_batch_size=True,
    resume_path=resume_path,
    hub_path_prefix=HUB_PATH_PREFIX,
    unify_ctc_tokenizer=True,  # AR decoder trains against the character
                                # tokenizer (recognizer/tokenizer/
                                # char_tokenizer.py) instead of the shared
                                # subword one -- see the markdown above, and
                                # the companion train_recognizer_ctc_only.ipynb,
                                # which uses the identical tokenizer so the two
                                # runs are directly comparable.
    train_ctc=False,  # CTC head fully excluded -- no forward pass, no loss term,
                       # no eval CER, no preview text. See model.forward's
                       # compute_ctc and compute_loss's docstring.
    train_ar=True,     # explicit for symmetry/clarity (this is already the default).
)


no Hub checkpoint found for Panhapich/tuna-ocr -- starting v2_ar_only from scratch
run_training: loading samples from /content/tuna-ocr/real_data/samples/dedup_filtered.arrow...
run_training: loaded 404912 samples, shuffling/splitting...
run_training: building char vocab...
run_training: scanning for unlearnable samples (CTC target > encoder frames)...
run_training: dropped 0 train / 0 val unlearnable samples in 38.5s (none)
run_training: building model + moving to cuda...


2026-08-26 09:04:10 run 'v2_ar_only': device=cuda, 396814 train / 8098 val samples, checkpoints -> /content/drive/My Drive/tuna-ocr/checkpoints/v2_ar_only, log file -> /content/drive/My Drive/tuna-ocr/checkpoints/v2_ar_only/train.log
2026-08-26 09:04:42 image widths ready in 32.2s (cache: /content/tuna-ocr/real_data/samples/widths_cache.json)
2026-08-26 09:04:49 auto batch size: 32 (probed mode(s): sequential, blockwise)
2026-08-26 09:04:49 batch_size=32, ~12400 steps/epoch, max_steps=80000 (~6.5 epochs)
2026-08-26 09:04:55 AR decoder training curriculum: sequential mode for steps 0-80000 (plain teacher forcing, unrestricted cross-attention -- see modules/decoder.py), then blockwise for the rest.
2026-08-26 09:05:18 step 100 epoch 0.01 loss 4.8884 ctc 20.1630 ce 4.8884 lr 1.35e-05 (22.5s)
2026-08-26 09:05:41 step 200 epoch 0.02 loss 4.2622 ctc 28.1012 ce 4.2622 lr 2.60e-05 (45.4s)
2026-08-26 09:06:03 step 300 epoch 0.02 loss 3.6569 ctc 37.1532 ce 3.6569 lr 3.84e-05 (67.3s)
2026-08-26 0

KeyboardInterrupt: 

## 4. Check progress

Reads this run's own `eval_log.csv` (columns `step,epoch,val_ar_cer,val_ctc_cer`) and prints the best/latest numbers so far. Safe to run anytime, including mid-run in a separate cell execution -- it only reads the log file.


In [ ]:
import pandas as pd

log_path = Path(checkpoint_root) / RUN_NAME / "eval_log.csv"
if not log_path.exists():
    print(f"no eval_log.csv yet at {log_path} -- has training started?")
else:
    df = pd.read_csv(log_path)
    latest = df.iloc[-1]
    best_ctc = df.loc[df["val_ctc_cer"].idxmin()]
    best_ar = df.loc[df["val_ar_cer"].idxmin()]
    print(f"{RUN_NAME}: {len(df)} eval points, latest step {int(latest.step)} (epoch {latest.epoch:.2f})")
    print(f"  latest:   val_ctc_cer {latest.val_ctc_cer:.4f}, val_ar_cer {latest.val_ar_cer:.4f}")
    print(f"  best ctc: {best_ctc.val_ctc_cer:.4f} at step {int(best_ctc.step)}")
    print(f"  best ar:  {best_ar.val_ar_cer:.4f} at step {int(best_ar.step)}")
